In [ ]:
import pickle
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.plots as plots
from python.error_metrics import LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies
from python.statistical_test import slope_ttest, page_trend_test, sign_test, wilcoxon_test, TrendType, test_stats_matrix
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS
from python.aggregation import TOP_LEVEL, BOTTOM_LEVEL, Aggregation

from itertools import product

In [ ]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=42)

if cfg.is_single_step_hierarchies():
    sh = SingleStepHierarchies(AVAILABLE_LEVELS)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

datasets = ['web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

In [ ]:
bottom_results = {}
top_results = {}
for d, m in tqdm.tqdm(product(datasets, models)):
    ID = f"{d}/{m}"
    tqdm.tqdm.write(ID)
    em = FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{cfg.MASTER_FOLDER}/{ID}/sample_hiers_errors.csv"), MI(error_metric='nrmse'))
    top_results[ID] = sh.error_matrix(em, sh.index_matrix(granularity=TOP_LEVEL, forecast_type=RECONCILED_FORECASTS))
    bottom_results[ID] = sh.error_matrix(em, sh.index_matrix(granularity=BOTTOM_LEVEL,   forecast_type=RECONCILED_FORECASTS))

with open(f"metrics/{cfg.MASTER_FOLDER}/bottom_error_matrices.pkl", 'wb') as f:
    pickle.dump(bottom_results, f)

with open(f"metrics/{cfg.MASTER_FOLDER}/top_error_matrices.pkl", 'wb') as f:
    pickle.dump(top_results, f)


In [ ]:
with open(f"metrics/{cfg.MASTER_FOLDER}/bottom_error_matrices.pkl", 'rb') as f:
    bottom_results = pickle.load(f)

with open(f"metrics/{cfg.MASTER_FOLDER}/top_error_matrices.pkl", 'rb') as f:
    top_results = pickle.load(f)

In [ ]:
data = top_results
test_fun = wilcoxon_test

pd.DataFrame(
    {
        m: {
            d: test_stats_matrix(data[f"{d}/{m}"], 'winsorized', TrendType.DECREASING, test_fun).p_value
            for d in datasets
        }
        for m in models
    }
)


In [ ]:
model = cfg.MODEL
dataset = cfg.DATASET.NAME
stat = 'winsorized'
data =  top_results # bottom_results

plots.plot_sampled_error_matrix(sh, stat, stats_matrix=data[f"{dataset}/{model}"])